# Chapter 04. 딥러닝 시작

In [1]:
import torch

## 활성화 함수


소프트맥스 함수

In [2]:
class Net(torch.nn.Module):
  def __init__(self, n_feature, n_hidden, n_output):
    super(Net, self).__init__()
    self.hidden = torch.nn.Linear(n_feature, n_hidden) # 은닉층
    self.relu = torch.nn.ReLu(inplace=True)
    self.out = torch.nn.Linear(n_hidden, n_output) # 출력층
    self.softmax = torch.nn.Softmax(dim=n_output)
  def forward(self, x):
    x = self.hidden(x)
    x = self.relu(x) # 은닉층을 위한 렐루 활성화 함수
    x = self.out(x)
    x = self.softmax(x) # 출력층을 위한 소프트맥스 활성화 함수
    return x

## 손실 함수

1. 평균 제곱 오차

In [3]:
loss_fn = torch.nn.MSELoss(reduction='sum')
"""모델 정의되지 않음"""
#y_pred = model(x)
#loss = loss_fn(y_pred, y)

'모델 정의되지 않음'

2. 크로스 엔트로피 오차

In [4]:
loss = torch.nn.CrossEntropyLoss()
input = torch.randn(5, 6, requires_grad = True) # torch.randn은 평균이 0이고 표준편차가 1인 가우시안 정규분포를 이용하여 숫자를 생성
target = torch.empty(5, dtype = torch.long).random_(6) # torch.empty는 dtype torch.float32의 랜덤한 값으로 채워진 텐서를 반환
output = loss(input, target)
output.backward()

## 딥러닝 문제점과 해결 방안

### 과적합 문제 발생
드롭아웃: 신경망 모델이 과적합되는 것을 피하기 위한 방법, 학습 과정 중 임의로 일부 노드들을 학습에서 제외시킨다.

In [5]:
import torch.nn.functional as F
class DropoutModel(torch.nn.Module):
  def __init__(self):
    super(DropoutModel, self).__init__()
    self.layer1 = torch.nn.Linear(784, 1200)
    self.dropout1 = torch.nn.ReLu(0.5) # 50%의 노드를 무작위로 선택하여 사용하지 않겠다는 의미
    self.layer2 = torch.nn.Linear(1200, 1200)
    self.dropout2 = torch.nn.ReLu(0.5)
    self.layer3 = torch.nn.Linear(1200, 10)

  def forward(self, x):
    x = F.relu(self.layer1(x))
    x = self.dropout1(x)
    x = F.relu(self.layer2(x))
    x = self.dropout2(x)
    return self.layer3(x)

미니 배치 경사 하강법

In [6]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
  def __init__(self):
    self.x_data = [[1,2,3], [4,5,6], [7,8,9]]
    self.y_data = [[12], [18], [11]]
  def __len__(self):
      return len(self.x_data)
  def __getitem__(self, idx):
      x = torch.FloatTensor(self.x_data[idx])
      y = torch.FloatTensor(self.y_data[idx])
      return x, y

dataset = CustomDataset()
dataloader = DataLoader(
    dataset, # 데이터셋
    batch_size=2, # 미니 배치 크기로 2의 제곱수를 사용하겠다는 의미
    shuffle=True, # 데이터를 불러올 때마다 랜덤으로 섞어서 가져오기
)